In [4]:
import pandas as pd
#opening the data file in read mode
df = pd.read_csv(r'D:\Research\Deep Learning and Generative AI\Module 3 - Natural Language Processing\Class 14_ Word Embeddings and Representations\sentiment-analysis-using-word2vec\Sentiment Analysis Dataset.tsv', sep='\t')
print(df)

            id  sentiment                                             review
0       5814_8          1  With all this stuff going down at the moment w...
1       2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2       7759_3          0  The film starts with a manager (Nicholas Bell)...
3       3630_4          0  It must be assumed that those who praised this...
4       9495_8          1  Superbly trashy and wondrously unpretentious 8...
...        ...        ...                                                ...
24995   3453_3          0  It seems like more consideration has gone into...
24996   5064_1          0  I don't believe they made this film. Completel...
24997  10905_3          0  Guy is a loser. Can't get girls, needs to buil...
24998  10194_3          0  This 30 minute documentary Buñuel made in the ...
24999   8478_8          1  I saw this movie as a child and it broke my he...

[25000 rows x 3 columns]


In [5]:
reviews = df.iloc[:, 2].values
labels = df.iloc[:, 1].values

In [6]:
# harsing html
from bs4 import BeautifulSoup
import re

def parseHtml(html):
    soup = BeautifulSoup(html, 'html.parser')
    return soup.get_text()


def removeDigits(string):
    for i in range(10):
        string = string.replace(str(i), ' ')
    return string

# removing html
reviews = list(map(parseHtml, reviews))

# removing digits
reviews = list(map(removeDigits, reviews))


In [7]:
#tokenizing
import nltk
nltk.download('punkt')
tokenizedText = [nltk.word_tokenize(item) for item in reviews]


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
# removing punctation
punc = '''!()-[]{};;'"\, <>./?@#$%^&*_~'''
tokenizedText = [[word for word in review if word not in punc] for review in tokenizedText]

<>:2: SyntaxWarning: invalid escape sequence '\,'
<>:2: SyntaxWarning: invalid escape sequence '\,'
C:\Users\User\AppData\Local\Temp\ipykernel_13860\2232155126.py:2: SyntaxWarning: invalid escape sequence '\,'
  punc = '''!()-[]{};;'"\, <>./?@#$%^&*_~'''


In [9]:
# Get total number of samples
totalRows = len(tokenizedText)

# Set train/test split ratio
splitRatio = 0.75
splitPoint = int(splitRatio * totalRows)

# Split the data
trainReviews = tokenizedText[:splitPoint]
testReviews = tokenizedText[splitPoint:]

# Split the labels (assuming `labels` is also a list and same length)
trainLabels = labels[:splitPoint]
testLabels = labels[splitPoint:]


In [15]:
from gensim.models import Word2Vec, KeyedVectors
import nltk

embeddingsSize=128
model = Word2Vec(trainReviews, vector_size=128, window=5, min_count=1, workers=4)


import numpy as np
def getVectors(dataset):
    vectors = []
    for dataItem in dataset:
        wordCount = 0
        singleDataItemEmbeddings = np.zeros(embeddingsSize)  # initialize before loop
        for word in dataItem:
            if word in model.wv.key_to_index:
                singleDataItemEmbeddings += model.wv[word]
                wordCount += 1

        if wordCount > 0:
            singleDataItemEmbeddings = singleDataItemEmbeddings / wordCount
        vectors.append(singleDataItemEmbeddings)
    return vectors

trainReviewVectors=getVectors(trainReviews)
testReviewVectors=getVectors(testReviews)

In [16]:


#Let's define a function that can display the accuracy, F1-score, label-wise precision, recall, etc. of each classifier

from sklearn.metrics import accuracy_score

from visualization import plot_confusion_matrix_from_data
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.metrics import f1_score

def printResults(y_true, y_predicted):
  print("Accuracy= ", accuracy_score(y_true, y_predicted))

  columns=['false', 'true']
  plot_confusion_matrix_from_data(y_true, y_predicted, columns)

  precision, recall, fscore, support = score(y_true, y_predicted)

  print('###########################################')
  print('precision: {}'.format(precision))
  print('recall: {}'.format(recall))
  print('fscore: {}'.format(fscore))
  print('support: {}'.format(support))
  print('###########################################3')

  print('Macro F1 ',f1_score(y_true, y_predicted, average='macro'))

  print('Micro F1 ', f1_score(y_true, y_predicted, average='micro'))


ModuleNotFoundError: No module named 'visualization'